In [2]:
import os
import time
import torch
import torch.nn as nn
import torch.optim
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader 
from chess import pgn
import chess
from tqdm import tqdm
from pathlib import  Path
from data_extraction import  board_to_tensor
from sklearn.model_selection import train_test_split
from torch.utils.data import random_split


BATCH_SIZE = 500000
TOTAL_SIZE = BATCH_SIZE*12

NUM_BATCH = 12


In [7]:
X_chunks = []
Y_chunks = []

for chunk_id in range(12):
    X_chunks.append(torch.load(fr"D:\Coding\CNN Chess Engine\data\gmdata\X_{chunk_id+1}.pt"))
    Y_chunks.append(torch.load(fr"D:\Coding\CNN Chess Engine\data\gmdata\Y_{chunk_id+1}.pt"))

X = torch.cat(X_chunks,dim=0)
Y = torch.cat(Y_chunks,dim=0)

In [5]:
from model import ChessCNN
move_to_idx =torch.load(r"D:\Coding\CNN Chess Engine\data\gmdata\move_to_idx.pt")

idx_to_move = torch.load(r"D:\Coding\CNN Chess Engine\data\gmdata\idx_to_move.pt")

num_moves = len(move_to_idx)
device = "cuda"

model = ChessCNN(num_moves).to(device)

In [8]:
dataset = TensorDataset(X, Y)

train_size = int(0.95*len(dataset))
val_size = len(dataset) - train_size

train_dataset , val_dataset = random_split(dataset,[train_size,val_size])

In [9]:
def train_one_epoch(model,loader,optimizer,criterion,device):
    
    model.train()

    running_loss = 0.0
    correct =0
    total =0
    for boards,targets in tqdm(loader, desc="Training"):
        boards = boards.float().to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        predictions = model(boards)
        predicted = predictions.argmax(dim=1)

        correct += (predicted == targets).sum().item()

        total += targets.size(0)
        
        loss = criterion(predictions,targets)
        
        

        loss.backward()

        optimizer.step()
        
        running_loss+=loss.item()

    avg_loss = running_loss/len(loader)
    accuracy = correct / total

    
    return avg_loss,accuracy

def evaluate(model,loader,criterion,device):
    model.eval()

    running_loss =0
    correct =0
    total =0
    with torch.no_grad():

        for boards,targets in loader:
            boards = boards.float().to(device)
            targets = targets.to(device)

            predictions = model(boards)
            predicted = predictions.argmax(dim=1)

            correct += (predicted == targets).sum().item()

            total += targets.size(0)
            loss = criterion(predictions,targets)

            running_loss +=loss.item()
    
    avg_loss = running_loss/len(loader)
    accuracy = correct / total

    return avg_loss,accuracy

In [10]:
train_loader = DataLoader(train_dataset,batch_size=256,shuffle=True,num_workers=0)
test_loader = DataLoader(val_dataset,batch_size=256,shuffle=True,num_workers=0)


In [ ]:

optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

criterion = nn.CrossEntropyLoss()

best_val_loss = float("inf")

In [ ]:
start_time = time.time()

num_epochs = 100

best_epoch = -1
best_val_loss = float("inf")

# checkpoint = torch.load(r"d:\Coding\CNN Chess Engine\data\gmdata\model_data\best_model.pth",map_location=device)
# model.load_state_dict(checkpoint["model_state_dict"])
# best_val_loss = checkpoint["val_loss"]

for epoch in range(num_epochs):

    epoch_start = time.time()

    print(f"\nEpoch {epoch+1}/{num_epochs}")

    train_loss, train_accuracy = train_one_epoch(model,train_loader,optimizer,criterion,device)

    val_loss, val_accuracy = evaluate(model,test_loader,criterion,device)

    torch.save(model.state_dict(),fr"d:\Coding\CNN Chess Engine\data\gmdata\model_data\epoch_{epoch+1}.pth")

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch + 1

        torch.save(
            {
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "train_accuracy": train_accuracy,
                "val_accuracy": val_accuracy,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
            },
            r"d:\Coding\CNN Chess Engine\data\gmdata\model_data\best_model.pth"
        )

        print(f"Saved Best Model (Epoch {epoch+1})")

    epoch_time = time.time() - epoch_start

    print(f"Train Loss     : {train_loss:.4f}")
    print(f"Train Accuracy : {train_accuracy:.4%}")

    print(f"Val Loss       : {val_loss:.4f}")
    print(f"Val Accuracy   : {val_accuracy:.4%}")

    print(f"Epoch Time     : {epoch_time:.2f}s")

total_time = time.time() - start_time

print(f"Total Time      : {total_time/60:.2f} min")
print(f"Avg Epoch Time  : {total_time/num_epochs:.2f} s")


Epoch 1/100


Training: 100%|██████████| 22266/22266 [08:15<00:00, 44.93it/s]


Saved Best Model (Epoch 1)
Train Loss     : 3.3311
Train Accuracy : 24.7620%
Val Loss       : 2.5383
Val Accuracy   : 33.4527%
Epoch Time     : 505.73s

Epoch 2/100


Training: 100%|██████████| 22266/22266 [08:19<00:00, 44.58it/s]


Saved Best Model (Epoch 2)
Train Loss     : 2.5585
Train Accuracy : 33.3372%
Val Loss       : 2.2979
Val Accuracy   : 36.5947%
Epoch Time     : 509.83s

Epoch 3/100


Training:  43%|████▎     | 9591/22266 [03:44<04:36, 45.88it/s]